In [2]:
from google.colab import drive
drive.mount('/content/mydrive')

Mounted at /content/mydrive


In [3]:
import os
repo_path = '/content/optimized-summarization'
if not os.path.exists(repo_path):
    !git clone https://github.com/srinisvas/optimized-summarization.git
else:
    print("Repo already exists, skipping clone.")

# Check files inside
os.listdir(repo_path)

Cloning into 'optimized-summarization'...
remote: Enumerating objects: 1447, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (211/211), done.
remote: Total 1447 (delta 129), reused 0 (delta 0), pack-reused 1236 (from 1)
Receiving objects: 100% (1447/1447), 107.86 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (434/434), done.


['.git', 'optimized-summarization', 'README.md', '.idea']

In [5]:
import os
import json
import time
import torch
from typing import Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- MODEL CONFIGURATION ---
MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"

try:
    print(f"Loading Model: {MODEL_NAME} for Ablation Study...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# --- DIRECTORY CONFIGURATION ---
NORMALIZED_DIR = '/content/optimized-summarization/optimized-summarization/Normalized-papers'
KEYPHRASE_DIR = '/content/optimized-summarization/optimized-summarization/salient-sentence-extraction/KeyPhrase Extraction'
BERTSUM_DIR = '/content/optimized-summarization/optimized-summarization/salient-sentence-extraction/Updated_BertSum'
OUTPUT_DIR = '/content/mydrive/MyDrive/NLP Project/Ablation_Sparse_Only_Local/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- LLM Generation Function ---

def call_local_llm(prompt: str, max_retries: int = 3) -> Optional[str]:
    """Generates text locally without chat-template overhead for pure ablation."""
    for attempt in range(max_retries):
        try:
            # We move directly to input_ids using the raw prompt string
            input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **input_ids,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Remove the prompt from the output to get only the generation
            clean_text = generated_text[len(prompt):].strip()
            return clean_text

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Generation failed after retries: {e}")
                return None
    return None

# --- Data Loading (Kept standard for Sparse Input extraction) ---

def extract_section_content(paper_data: Dict[str, Any], keywords: list) -> str:
    content = []
    for section in paper_data.get('sections', []):
        title = section.get('title', '').lower().strip()
        if any(kw in title for kw in keywords):
            content.append(section.get('content', ''))
    return "\n".join(item for item in content if item).strip()

def load_paper_data(file_name: str) -> Optional[Dict[str, Any]]:
    try:
        with open(os.path.join(NORMALIZED_DIR, file_name), 'r', encoding='utf-8') as f:
            paper_data = json.load(f)
        with open(os.path.join(BERTSUM_DIR, file_name), 'r', encoding='utf-8') as f:
            salient = json.load(f).get('salient_sentences', [])
        with open(os.path.join(KEYPHRASE_DIR, file_name), 'r', encoding='utf-8') as f:
            phrases = json.load(f).get('combined_deduplicated', [])

        return {
            "paper_id": file_name.replace('.json', ''),
            "abstract": extract_section_content(paper_data, ["abstract"]),
            "conclusion": extract_section_content(paper_data, ["conclusion", "summary", "discuss"]),
            "salient_sentences": salient,
            "keyphrases": phrases
        }
    except Exception as e:
        print(f"Loading error for {file_name}: {e}")
        return None

# --- ABLATION PROMPT (No Logic, Just Sparse Data) ---

def create_sparse_only_prompt(data: Dict[str, Any]) -> str:
    """
    Constructs a raw prompt with ONLY the sparse input data.
    All system instructions, examples, and CoT are removed.
    """
    prompt = f"""Summarize the following research paper data:

ABSTRACT:
{data['abstract']}

CONCLUSION:
{data['conclusion']}

IMPORTANT SENTENCES:
- {"\n- ".join(data['salient_sentences'])}

KEY PHRASES:
{', '.join(data['keyphrases'])}

SUMMARY:"""
    return prompt

# --- Main Loop ---

def run_orchestration():
    print("RUNNING ABLATION STUDY: Sparse Input Only (No Adaptive Prompting)")

    file_list = sorted([f for f in os.listdir(NORMALIZED_DIR) if f.endswith('.json')])
    processed_count = 0

    for file_name in file_list:
        paper_id = file_name.replace('.json', '')
        print(f"\nProcessing {paper_id} (Sparse Only)...")

        data = load_paper_data(file_name)
        if not data: continue

        # Generate the raw data-only prompt
        raw_prompt = create_sparse_only_prompt(data)

        # Call LLM with the raw string
        summary_text = call_local_llm(raw_prompt)

        if summary_text:
            output_path = os.path.join(OUTPUT_DIR, f"{paper_id}.txt")
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(summary_text)
            print(f"Saved: {output_path}")
            processed_count += 1

    print(f"\nStudy complete. {processed_count} summaries generated using ONLY sparse input.")

if __name__ == "__main__":
    run_orchestration()

Loading Model: HuggingFaceTB/SmolLM3-3B for Ablation Study...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Model loaded successfully.
RUNNING ABLATION STUDY: Sparse Input Only (No Adaptive Prompting)

Processing A Bibliometric View of AI Ethics Development (Sparse Only)...
Saved: /content/mydrive/MyDrive/NLP Project/Ablation_Sparse_Only_Local/A Bibliometric View of AI Ethics Development.txt

Processing A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI (Sparse Only)...
Saved: /content/mydrive/MyDrive/NLP Project/Ablation_Sparse_Only_Local/A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI.txt

Processing A Privacy Impact Assessment Tool for Cloud Computing (Sparse Only)...
Saved: /content/mydrive/MyDrive/NLP Project/Ablation_Sparse_Only_Local/A Privacy Impact Assessment Tool for Cloud Computing.txt

Processing A Privacy Maturity Model for Cloud Storage Services (Sparse Only)...
Saved: /content/mydrive/MyDrive/NLP Project/Ablation_Sparse_Only_Local/A Privacy Maturity Model for Cloud Storage Services.txt

Processing A Privacy-Leakage-Tol

KeyboardInterrupt: 

In [4]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 43.5 MB/s eta 0:00:00
